# RAG Query & Testing

This notebook is for **testing and querying** the deployed RAG system.

**Prerequisites:**
- Run `rag_pipeline_build.ipynb` first to deploy the pipeline
- This notebook must run **inside the cluster** (RHOAI notebook server) since it connects to cluster-internal services (Milvus, vLLM, embedding service)

## What This Notebook Covers

1. **Verify Services** — Check Milvus, LLM, and embedding endpoints
2. **Test RAG Queries** — End-to-end retrieval-augmented generation
3. **Inspect RAG Process** — Explore chunks, prompts, and comparisons
4. **Interactive Q&A** — Live query loop

In [ ]:
# Install required packages
!pip install -q pymilvus sentence-transformers openai requests

---
## 1. Configuration

These values must match what was used in `rag_pipeline_build.ipynb`.

In [ ]:
NAMESPACE = "ray-docling"

# Milvus
MILVUS_HOST = "milvus-milvus.milvus.svc.cluster.local"
MILVUS_PORT = 19530
COLLECTION_NAME = "open_rag_pdfs"

# Embedding
EMBEDDING_MODE = "local"  # Must match pipeline: "local" or "service"
EMBEDDING_MODEL = "ibm-granite/granite-embedding-125m-english"
EMBEDDING_DIM = 768
EMBEDDING_SERVICE_NAME = "embedding-service"
EMBEDDING_ENDPOINT = f"http://{EMBEDDING_SERVICE_NAME}.{NAMESPACE}.svc.cluster.local:8080"

# LLM
# InferenceService name (derived from model name by model_deployment component)
LLM_ISVC_NAME = "mistral-7b-instruct-v0-3"
# KServe RawDeployment auto-creates a predictor service: {isvc-name}-predictor
INFERENCE_URL = f"http://{LLM_ISVC_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8080/v1"

print(f"Milvus:         {MILVUS_HOST}:{MILVUS_PORT}")
print(f"Collection:     {COLLECTION_NAME}")
print(f"Embedding mode: {EMBEDDING_MODE}")
if EMBEDDING_MODE == "service":
    print(f"Embedding URL:  {EMBEDDING_ENDPOINT}")
else:
    print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LLM ISVC:       {LLM_ISVC_NAME}")
print(f"LLM endpoint:   {INFERENCE_URL}")

---
## 2. Verify Services

### 2.1 Milvus Collection

In [ ]:
from pymilvus import MilvusClient, connections, Collection

milvus_uri = f"http://{MILVUS_HOST}:{MILVUS_PORT}"
print(f"Connecting to {milvus_uri}")

milvus_client = MilvusClient(uri=milvus_uri, db_name="default")
connections.connect(alias="default", host=MILVUS_HOST, port=str(MILVUS_PORT))

if not milvus_client.has_collection(COLLECTION_NAME):
    print(f"❌ Collection '{COLLECTION_NAME}' not found.")
    print("Available:", milvus_client.list_collections())
    print("The ingestion step may not have completed yet.")
else:
    collection = Collection(COLLECTION_NAME)
    collection.flush()
    milvus_client.load_collection(COLLECTION_NAME)
    print(f"✅ Collection: {COLLECTION_NAME}")
    print(f"   Vectors:    {collection.num_entities}")

In [ ]:
# Sample chunks
samples = milvus_client.query(
    collection_name=COLLECTION_NAME,
    filter="chunk_index >= 0",
    output_fields=["source_file", "chunk_index", "text"],
    limit=5,
)

print(f"Sample chunks ({len(samples)}):\n")
for s in samples:
    print(f"  [{s['source_file']} | chunk {s['chunk_index']}]")
    print(f"  {s['text'][:200]}...")
    print()

### 2.2 LLM Endpoint

In [ ]:
import requests

print(f"Checking: {INFERENCE_URL}")

try:
    resp = requests.get(f"{INFERENCE_URL}/models", timeout=10)
    resp.raise_for_status()
    models = resp.json()
    print(f"\n✅ Available models:")
    for m in models.get("data", []):
        print(f"  - {m['id']}")
except requests.ConnectionError:
    print("❌ Connection failed. The model may not be deployed yet.")
except Exception as e:
    print(f"❌ Error: {e}")

### 2.3 Embedding (mode-dependent)

In [ ]:
if EMBEDDING_MODE == "service":
    print(f"Testing embedding service: {EMBEDDING_ENDPOINT}")
    try:
        resp = requests.post(
            f"{EMBEDDING_ENDPOINT}/v1/embeddings",
            json={"model": EMBEDDING_MODEL, "input": ["test"]},
            timeout=10
        )
        resp.raise_for_status()
        dim = len(resp.json()['data'][0]['embedding'])
        print(f"✅ Embedding service ready (dim={dim})")
    except Exception as e:
        print(f"❌ Embedding service error: {e}")
else:
    print(f"Loading local model: {EMBEDDING_MODEL}")
    from sentence_transformers import SentenceTransformer
    _test_model = SentenceTransformer(EMBEDDING_MODEL)
    _test_emb = _test_model.encode(["test"], normalize_embeddings=True)
    print(f"✅ Local model ready (dim={_test_emb.shape[1]})")
    del _test_model, _test_emb

---
## 3. Initialize RAG Components

In [ ]:
from openai import OpenAI
import requests

# Embedding setup
embed_model = None
embedding_service_url = None

if EMBEDDING_MODE == "local":
    from sentence_transformers import SentenceTransformer
    embed_model = SentenceTransformer(EMBEDDING_MODEL)
    print(f"Embedding: local ({EMBEDDING_MODEL}, dim={EMBEDDING_DIM})")
else:
    embedding_service_url = EMBEDDING_ENDPOINT
    print(f"Embedding: service ({embedding_service_url})")

# LLM setup
llm_client = OpenAI(base_url=INFERENCE_URL, api_key="unused")
print(f"LLM: {INFERENCE_URL}")
print("\n✅ RAG components initialized")

In [ ]:
def get_embedding(text):
    """Generate embedding using the configured mode."""
    if EMBEDDING_MODE == "local":
        return embed_model.encode([text], normalize_embeddings=True).tolist()[0]
    else:
        resp = requests.post(
            f"{embedding_service_url}/v1/embeddings",
            json={"model": EMBEDDING_MODEL, "input": [text]},
            timeout=30
        )
        resp.raise_for_status()
        return resp.json()["data"][0]["embedding"]


def search_similar_chunks(question, top_k=5):
    """Embed the question and search Milvus for similar chunks."""
    query_embedding = get_embedding(question)

    results = milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=[query_embedding],
        limit=top_k,
        output_fields=["source_file", "chunk_index", "text"],
        search_params={"metric_type": "COSINE", "params": {"nprobe": 16}},
    )

    chunks = []
    for hits in results:
        for hit in hits:
            chunks.append({
                "text": hit["entity"]["text"],
                "source_file": hit["entity"]["source_file"],
                "chunk_index": hit["entity"]["chunk_index"],
                "score": hit["distance"],
            })
    return chunks


def build_rag_prompt(question, chunks):
    """Build a prompt with retrieved context for the LLM."""
    context_block = "\n\n---\n\n".join(
        f"[Source: {c['source_file']}, chunk {c['chunk_index']}, "
        f"similarity: {c['score']:.3f}]\n{c['text']}"
        for c in chunks
    )
    return (
        "You are a helpful assistant. Answer the user's question based on the "
        "provided context documents. If the answer is not in the context, say so.\n\n"
        f"## Context\n\n{context_block}\n\n"
        f"## Question\n\n{question}\n\n"
        "## Answer\n\n"
    )


def ask_llm(prompt, max_tokens=512, temperature=0.1):
    """Send the prompt to vLLM and return the response."""
    response = llm_client.chat.completions.create(
        model=LLM_ISVC_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content


def rag_query(question, top_k=5, max_tokens=512, temperature=0.1):
    """End-to-end RAG: retrieve context -> build prompt -> generate answer."""
    chunks = search_similar_chunks(question, top_k=top_k)
    prompt = build_rag_prompt(question, chunks)
    answer = ask_llm(prompt, max_tokens=max_tokens, temperature=temperature)

    return {
        "question": question,
        "answer": answer,
        "sources": chunks,
        "prompt": prompt,
    }


print("✅ RAG functions ready")

---
## 4. Test RAG Queries

In [ ]:
question = "What are the main topics covered in the documents?"

result = rag_query(question)

print(f"Question: {result['question']}\n")
print(f"Answer:\n{result['answer']}\n")
print(f"Sources ({len(result['sources'])}):\n")
for s in result["sources"]:
    print(f"  - {s['source_file']} (chunk {s['chunk_index']}, similarity: {s['score']:.3f})")

In [ ]:
# Try another question
result = rag_query("Summarize the key findings.")

print(f"Question: {result['question']}\n")
print(f"Answer:\n{result['answer']}\n")
print(f"Sources ({len(result['sources'])}):\n")
for s in result["sources"]:
    print(f"  - {s['source_file']} (chunk {s['chunk_index']}, similarity: {s['score']:.3f})")

---
## 5. Inspect the RAG Process

Inspect each step independently to debug or understand the pipeline.

### 5.1 Retrieval Only — View Chunks

In [ ]:
question = "What are the main topics covered in the documents?"
chunks = search_similar_chunks(question, top_k=5)

print(f"Top {len(chunks)} chunks for: '{question}'\n")
for i, c in enumerate(chunks, 1):
    print(f"--- Chunk {i} (score: {c['score']:.3f}) ---")
    print(f"Source: {c['source_file']}, chunk_index: {c['chunk_index']}")
    print(c["text"][:500])
    print()

### 5.2 Inspect the Prompt

In [ ]:
prompt = build_rag_prompt(question, chunks)
print(f"Prompt length: {len(prompt)} characters\n")
print(prompt)

### 5.3 RAG vs. Direct LLM — Side-by-Side Comparison

In [ ]:
# RAG answer
rag_result = rag_query(question)

# Direct LLM answer (no context)
direct_response = llm_client.chat.completions.create(
    model=LLM_ISVC_NAME,
    messages=[{"role": "user", "content": question}],
    max_tokens=512,
    temperature=0.1,
)
direct_answer = direct_response.choices[0].message.content

print(f"Question: {question}")
print("\n" + "=" * 60)
print("WITH RAG context:")
print("=" * 60)
print(rag_result['answer'])

print("\n" + "=" * 60)
print("WITHOUT RAG context (direct LLM):")
print("=" * 60)
print(direct_answer)

print("\n" + "=" * 60)
print("The RAG answer references specific document content,")
print("while the direct answer is generic.")

---
## 6. Interactive Query Loop

Type questions and get RAG-powered answers. Type `quit` to exit.

In [ ]:
while True:
    question = input("\nAsk a question (or 'quit'): ").strip()
    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not question:
        continue

    result = rag_query(question)
    print(f"\nAnswer:\n{result['answer']}\n")
    print(f"Sources:")
    for s in result["sources"]:
        print(f"  - {s['source_file']} (chunk {s['chunk_index']}, {s['score']:.3f})")

---
## 7. Cleanup

Uncomment to tear down resources.

In [ ]:
# Drop the Milvus collection
# milvus_client.drop_collection(COLLECTION_NAME)
# print(f"Dropped collection: {COLLECTION_NAME}")

# Close Milvus connection
# connections.disconnect("default")
# print("Disconnected from Milvus")